# 5-fold 결과 복구: mAP=-1 및 alpha 선택 오류 수정

**재학습하지 않습니다.** 기존 5개 YOLO/CNN 가중치와 저장된 outer-holdout 예측을 재사용합니다.

기존 COCO 요약의 전체 AP는 기본 maxDets=100을 참조하는데 평가 설정은 [1,10,300]이어서 전체 map만 -1이 될 수 있습니다. 기존 코드는 유효하지 않은 map을 alpha 선택에 사용했고, 전부 -1이면 동점 규칙에 따라 alpha=0을 선택했습니다. 따라서 기존 hybrid 결과로 CNN 효과가 없다고 결론 내리면 안 됩니다.

이번 수정은 후보 수를 100으로 바꾸지 않고, 원래 의도한 최대 **300개**에서 COCO precision tensor를 직접 요약합니다. 전체 AP가 유효하지 않으면 alpha 선택을 중단합니다. AP50과 AP75도 같은 설정으로 다시 계산합니다.

절차: 원본·가중치 확인 → 기존 inner val 재추론 → 원래 alpha 후보군으로 검증 선택 → 설정 고정 → 저장된 outer 예측 재평가. 원본 test는 사용하지 않습니다. 5개의 평가 fold가 여기서 outer test입니다.


## v2: 박스 처리와 검증 캐시 복구

면적이 0인 유한한 예측 박스는 COCO에서 GT와 매칭되지 않는 예측으로 유지합니다. 점수·순위·탐지 상한 및 운영 임계값에 따라 FP에 포함될 수 있습니다. 이 예측을 삭제해서 성능을 올리지 않습니다. GT의 면적이 0이거나 좌표가 NaN/Inf이면 이미지 경로·GT/예측 구분·박스 번호·좌표를 표시하고 중단합니다. 역전된 예측 좌표도 중단합니다.

이전 부분 실행에서 저장된 검증 예측이 있으면, 가중치 hash·추론 패키지 버전·검증 이미지 구성·GT를 확인하고 재사용합니다. 학습은 반복하지 않습니다. 이번에 확인된 실제 경로 `.../week5_kfold_results/kfold5`를 기본값으로 넣었습니다.

먼저 기존 Colab에서 평가기만 바로 수정하려면 별도로 안내된 패치 셀을 사용할 수 있습니다. 이 노트북은 수정 모듈 두 개를 모두 포함한 완성본입니다.


## 1. 환경 준비

In [ ]:
# 가능하면 학습한 Colab 런타임에서 실행하세요. PyTorch는 여기서 재설치하지 않습니다.
%pip install -q "ultralytics==8.4.174" pycocotools pandas matplotlib
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')


## 2. 완료 결과 폴더 지정

In [ ]:
# 사용자가 보낸 완료 결과 경로를 기본값으로 설정했습니다.
RUN_ROOT = '/content/drive/MyDrive/Pothole/week5_kfold_results/kfold5'
DATA_ROOT = '/content/drive/MyDrive/Pothole'
ALLOW_VERSION_DRIFT = False  # 원 실행 torch/torchvision/ultralytics 버전과 다르면 중단
REMEASURE_LATENCY = False    # 먼저 정확도 복구; True면 같은 방식으로 처리시간도 다시 측정


## 3. 수정 평가기 작성

In [ ]:
%%writefile /content/week5_coco_fixed.py
"""COCO AP at the explicitly selected maxDets, without default-only summarize()."""
import contextlib, io
import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def evaluate(pred, rows, max_det=300):
    if len(pred)!=len(rows) or not rows: raise ValueError('Prediction/GT image counts must agree and be nonempty')
    if not isinstance(max_det,int) or max_det<=10: raise ValueError('max_det must be an integer >10')
    images=[{'id':i} for i in range(len(rows))]; cats=[{'id':1,'name':'pothole'}]
    ga=[]; da=[]
    for i,(p,r) in enumerate(zip(pred,rows)):
        gt=np.asarray(r['gt'],dtype=np.float64).reshape(-1,4)
        boxes=np.asarray(p['boxes'],dtype=np.float64).reshape(-1,4)
        scores=np.asarray(p['scores'],dtype=np.float64).reshape(-1)
        if len(boxes)!=len(scores) or not np.isfinite(scores).all(): raise ValueError('Bad prediction scores')
        for arr,target in [(gt,ga),(boxes,da)]:
            role='GT' if target is ga else 'prediction'
            image=r.get('path',r.get('image',f'image_index={i}'))
            if not np.isfinite(arr).all():
                bad=np.flatnonzero(~np.isfinite(arr).all(axis=1))
                raise ValueError(f'{role}: nonfinite bbox in {image}; indices={bad.tolist()}, boxes={arr[bad].tolist()}')
            wh=arr[:,2:]-arr[:,:2]
            bad=np.flatnonzero(np.any(wh<=0,axis=1) if target is ga else np.any(wh<0,axis=1))
            if len(bad):
                raise ValueError(f'{role}: invalid bbox geometry in {image}; indices={bad.tolist()}, boxes={arr[bad].tolist()}')
            # Finite zero-area predictions remain unmatched detections (FP) under COCO.
            # Do not silently remove them or change GT annotations.
            for j,b in enumerate(arr):
                x,y,x2,y2=map(float,b); w,h=x2-x,y2-y
                ann={'id':len(target)+1,'image_id':i,'category_id':1,'bbox':[x,y,w,h],'area':w*h,'iscrowd':0}
                if target is da: ann['score']=float(scores[j])
                target.append(ann)
    if not ga: raise ValueError('No ground truth; overall AP is undefined')
    with contextlib.redirect_stdout(io.StringIO()):
        gt=COCO(); gt.dataset={'images':images,'categories':cats,'annotations':ga,'info':{}}; gt.createIndex()
        dt=COCO(); dt.dataset={'images':images,'categories':cats,'annotations':da,'info':{}}; dt.createIndex()
        ev=COCOeval(gt,dt,'bbox'); ev.params.imgIds=list(range(len(rows))); ev.params.catIds=[1]
        ev.params.maxDets=[1,10,max_det]
        ev.evaluate(); ev.accumulate()
    # precision shape: IoU x recall x class x area x maxDets
    mi=ev.params.maxDets.index(max_det)
    def ap(area='all',iou=None):
        ai=ev.params.areaRngLbl.index(area)
        values=ev.eval['precision'][:,:,:,ai,mi]
        if iou is not None: values=values[np.isclose(ev.params.iouThrs,iou)]
        values=values[values>-1]
        return float(values.mean()) if values.size else -1.
    result={'map':ap(),'map_50':ap(iou=.5),'map_75':ap(iou=.75),
            'map_small':ap('small'),'map_medium':ap('medium'),'map_large':ap('large')}
    if any(not np.isfinite(result[k]) or not 0<=result[k]<=1 for k in ['map','map_50','map_75']):
        raise RuntimeError(f'Invalid overall COCO AP; refusing model selection: {result}')
    return result,ev


def map_metrics(pred,rows,cfg): return evaluate(pred,rows,int(cfg['max_det']))[0]


def select_alpha(trials):
    if not trials: raise ValueError('No alpha trials')
    for r in trials:
        if not np.isfinite(r['map']) or not 0<=r['map']<=1:
            raise ValueError(f'Invalid validation mAP; cannot select alpha: {r}')
    return max(trials,key=lambda x:(x['map'],-x['alpha']))


def self_test():
    b=np.array([[0.,0.,20.,20.]])
    p=[{'boxes':b,'scores':np.array([.9])}]; t=[{'gt':b}]
    for cap in [100,300]:
        m,ev=evaluate(p,t,cap); assert np.isclose(m['map'],1.) and np.isclose(m['map_50'],1.)
        with contextlib.redirect_stdout(io.StringIO()): ev.summarize()
        if cap==100: assert np.isclose(ev.stats[0],m['map'])
        elif ev.stats[0]==-1: print('Reproduced: default summarize returns -1 at maxDets=300; corrected AP=1.0')
    empty=[{'boxes':np.empty((0,4)),'scores':np.empty(0)}]
    assert evaluate(empty,t)[0]['map']==0
    # A high-score false positive before the true detection lowers AP.
    p2=[{'boxes':np.array([[30,30,50,50],[0,0,20,20]]),'scores':np.array([.95,.9])}]
    assert np.isclose(evaluate(p2,t)[0]['map'],.5)
    # Same prediction, IoU=.6: only .50/.55/.60 thresholds pass -> mAP=.3.
    p3=[{'boxes':np.array([[5,0,25,20]]),'scores':np.array([.9])}]
    assert np.isclose(evaluate(p3,t)[0]['map'],.3)
    assert select_alpha([{'alpha':0.,'map':.4},{'alpha':1.,'map':.6}])['alpha']==1.
    try: select_alpha([{'alpha':0.,'map':-1.}])
    except ValueError: pass
    else: raise AssertionError('Invalid AP must stop alpha selection')
    zero=[{'boxes':np.array([[10,10,10,20],[0,0,20,20]]),'scores':np.array([.95,.9])}]
    assert np.isclose(evaluate(zero,t)[0]['map'],.5)  # invalid-region detection still contributes FP
    assert evaluate([{'boxes':np.array([[10,10,10,20]]),'scores':np.array([.9])}],t)[0]['map']==0
    for bad_gt in [np.array([[0,0,0,20]]),np.array([[0,0,np.nan,20]])]:
        try: evaluate(p,[{'gt':bad_gt,'path':'bad_gt_image.jpg'}])
        except ValueError as exc: assert 'GT' in str(exc) and 'bad_gt_image.jpg' in str(exc)
        else: raise AssertionError('Invalid GT must not be removed')
    try: evaluate([{'boxes':np.array([[20,0,10,20]]),'scores':np.array([.9])}],t)
    except ValueError as exc: assert 'prediction' in str(exc)
    else: raise AssertionError('Reversed prediction must fail')
    print('PASS: AP100/AP300, empty detections, FP ordering, IoU averaging, zero-area prediction FP, GT diagnostic, invalid-AP guard')

if __name__=='__main__': self_test()


## 4. 재학습 없는 복구 코드 작성

In [ ]:
%%writefile /content/week5_recovery.py
from pathlib import Path
import csv, hashlib, json, time
import cv2
import numpy as np
import torch
from torch import nn
from ultralytics import YOLO
from week5_coco_fixed import map_metrics, select_alpha
def dump(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

def ious(a, b):
    a=np.asarray(a, dtype=np.float32).reshape(-1,4)
    b=np.asarray(b, dtype=np.float32).reshape(-1,4)
    lo=np.maximum(a[:,None,:2], b[None,:,:2]); hi=np.minimum(a[:,None,2:], b[None,:,2:])
    inter=np.maximum(hi-lo,0).prod(2)
    aa=np.maximum(a[:,2:]-a[:,:2],0).prod(1)
    bb=np.maximum(b[:,2:]-b[:,:2],0).prod(1)
    return inter / np.maximum(aa[:,None]+bb[None,:]-inter, 1e-9)

class CropCNN(nn.Module):
    def __init__(self):
        super().__init__()
        blocks=[]; c=3
        for width in [32,64,128]:
            blocks += [nn.Conv2d(c,width,3,padding=1),nn.BatchNorm2d(width),nn.ReLU(),nn.MaxPool2d(2)]
            c=width
        self.net=nn.Sequential(*blocks,nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(.2),nn.Linear(128,1))
    def forward(self,x): return self.net(x).flatten()

def crop_tensor(im,box,size):
    h,w=im.shape[:2]
    x1,y1=np.floor(box[:2]).astype(int); x2,y2=np.ceil(box[2:]).astype(int)
    x1,y1=max(0,min(x1,w-1)),max(0,min(y1,h-1))
    x2,y2=max(x1+1,min(x2,w)),max(y1+1,min(y2,h))
    crop=cv2.cvtColor(cv2.resize(im[y1:y2,x1:x2],(size,size)),cv2.COLOR_BGR2RGB)
    return torch.from_numpy(crop.transpose(2,0,1).copy()).float()/255.

@torch.inference_mode()
def cnn_scores(net,im,boxes,cfg):
    if len(boxes)==0: return np.empty(0,dtype=np.float32)
    result=[]
    for start in range(0,len(boxes),cfg['cnn_batch']):
        x=torch.stack([crop_tensor(im,b,cfg['crop_size']) for b in boxes[start:start+cfg['cnn_batch']]])
        result.append(net(x.to(cfg['torch_device'])).sigmoid().cpu().numpy())
    return np.concatenate(result)

def rescore(base,probs,alpha):
    return [{'boxes':p['boxes'],'scores':p['scores']*np.maximum(q,1e-8)**alpha} for p,q in zip(base,probs)]

def counts_at(pred,rows,threshold,iou=.5):
    result=[]
    for p,r in zip(pred,rows):
        idx=np.flatnonzero(p['scores']>=threshold)
        idx=idx[np.argsort(-p['scores'][idx],kind='stable')]
        overlap=ious(p['boxes'][idx],r['gt']); used=set(); tp=0
        for ov in overlap:
            eligible=[j for j in range(len(ov)) if j not in used and ov[j]>=iou]
            if eligible:
                j=max(eligible,key=lambda j:ov[j]); used.add(j); tp+=1
        result.append([tp,len(idx)-tp,len(r['gt'])-tp])
    return np.asarray(result,dtype=np.int64)

def operating_metrics(counts):
    tp,fp,fn=np.asarray(counts).sum(0)
    p=tp/max(tp+fp,1); r=tp/max(tp+fn,1)
    return {'precision':float(p),'recall':float(r),'f1':float(2*tp/max(2*tp+fp+fn,1)),
            'tp':int(tp),'fp':int(fp),'fn':int(fn),'fp_per_image':float(fp/len(counts))}

def tune_threshold(pred,rows,cfg):
    grid=np.unique(np.r_[cfg['proposal_conf'],np.linspace(.01,.99,99)])
    values=[operating_metrics(counts_at(pred,rows,float(t)))['f1'] for t in grid]
    # deterministic tie-breaking: smallest threshold, favor recall.
    i=int(np.argmax(values)); return float(grid[i]),float(values[i])

def sync(cfg):
    if cfg['torch_device'].startswith('cuda'): torch.cuda.synchronize()

def benchmark(det,net,rows,alpha,cfg):
    # Decode before timer; include preprocessing, YOLO, NMS, crop, CNN, score transfer.
    ims=[cv2.imread(r['path']) for r in rows[:cfg['benchmark_images']]]
    times={'YOLOv8':[],'YOLOv8+CNN':[]}
    def run(im,hybrid):
        p=predict_yolo(det,im,cfg)
        if hybrid:
            q=cnn_scores(net,im,p['boxes'],cfg); p['scores']=p['scores']*np.maximum(q,1e-8)**alpha
        return p
    for _ in range(3):
        for hybrid in [False,True]: run(ims[0],hybrid)
    for rep in range(cfg['benchmark_repeats']):
        order=[False,True] if rep%2==0 else [True,False]
        for im in ims:
            for hybrid in order:
                sync(cfg); start=time.perf_counter(); run(im,hybrid); sync(cfg)
                times['YOLOv8+CNN' if hybrid else 'YOLOv8'].append((time.perf_counter()-start)*1000)
    return {k:{'mean_ms':float(np.mean(v)),'median_ms':float(np.median(v)),'p95_ms':float(np.percentile(v,95)),
               'fps':float(1000/np.mean(v)),'samples':len(v),'scope':'in-memory single image; disk decoding excluded; FP32'} for k,v in times.items()}

def write_csv(path,rows):
    with open(path,'w',newline='',encoding='utf-8-sig') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
def predict_yolo(model,im,cfg):
    from ultralytics.cfg import DEFAULT_CFG_DICT
    precision={'quantize':32} if 'quantize' in DEFAULT_CFG_DICT else {'half':False}
    r=model.predict(im,imgsz=cfg['imgsz'],conf=cfg['proposal_conf'],iou=cfg['nms_iou'],
                    max_det=cfg['max_det'],device=cfg['device'],verbose=False,**precision)[0]
    return {'boxes':r.boxes.xyxy.cpu().numpy(),'scores':r.boxes.conf.cpu().numpy()}

@torch.inference_mode()
def collect(det,net,rows,cfg):
    base=[]; probs=[]
    for i,row in enumerate(rows,1):
        im=cv2.imread(row['path'])
        if im is None: raise ValueError(row['path'])
        p=predict_yolo(det,im,cfg); base.append(p); probs.append(cnn_scores(net,im,p['boxes'],cfg))
        if i%20==0 or i==len(rows): print(f'Validation inference: {i}/{len(rows)}',flush=True)
    return base,probs


def load_json(p): return json.loads(Path(p).read_text())


def check_versions(run_dir,allow_drift):
    import importlib.metadata as metadata
    expected={}
    for line in (run_dir/'pip_freeze.txt').read_text().splitlines():
        if '==' in line:
            k,v=line.split('==',1); expected[k.lower()]=v
    actual={k:metadata.version(k) for k in ['ultralytics','torch','torchvision']}
    mismatch={k:{'old':expected.get(k),'current':v} for k,v in actual.items() if expected.get(k)!=v}
    if mismatch and not allow_drift:
        raise RuntimeError(f'추론 재현을 위해 원 실행 버전이 필요합니다: {mismatch}. 원 Colab 환경을 사용하거나 해당 버전을 설치하세요. '
                           '버전 차이를 허용하는 별도 재평가는 ALLOW_VERSION_DRIFT=True로 명시하고 결과에 기록하세요.')
    return {'expected':{k:expected.get(k) for k in actual},'actual':actual,'mismatch':mismatch}


def validation_rows(source,indices,data_root,eps,old_manifest):
    """Reconstruct exact staged GT; check original and canonical-label hashes."""
    rows=[]
    staged={int(Path(r['path']).stem):r for r in old_manifest if r['split']=='val'}
    for idx in sorted(indices):
        src=source[idx]; p=Path(data_root)/src['relative_path']; label=p.parent.parent/'labels'/(p.stem+'.txt')
        if sha(label)!=src['label_sha256']: raise ValueError(f'학습 후 원본 라벨이 변경되었습니다: {label}')
        im=cv2.imread(str(p))
        if im is None: raise ValueError(f'원본 이미지 읽기 실패: {p}')
        h,w=im.shape[:2]
        pixel=hashlib.sha256(np.asarray(im.shape,dtype=np.int64).tobytes()+im.tobytes()).hexdigest()
        if pixel!=src['pixel_sha256']: raise ValueError(f'학습 후 이미지가 변경되었습니다: {p}')
        canonical=[]
        for line in label.read_text().splitlines():
            if not line.strip(): continue
            v=np.array(list(map(float,line.split())),dtype=np.float64)
            if len(v)!=5 or v[0]!=0 or not np.isfinite(v).all(): raise ValueError(f'라벨 오류: {label}')
            _,x,y,bw,bh=v
            if not (-eps<=x<=1+eps and -eps<=y<=1+eps and 0<bw<=1+eps and 0<bh<=1+eps): raise ValueError(f'좌표 오류: {label}')
            v[1:]=np.clip(v[1:],0,1); _,x,y,bw,bh=v
            box=np.clip([(x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h],[0,0,0,0],[w,h,w,h])
            x1,y1,x2,y2=box
            cv=[0,(x1+x2)/(2*w),(y1+y2)/(2*h),(x2-x1)/w,(y2-y1)/h]
            canonical.append(' '.join(f'{a:.12g}' for a in cv))
        txt='\n'.join(canonical)+'\n' if canonical else ''
        if hashlib.sha256(txt.encode()).hexdigest()!=staged[idx]['label_sha256']:
            raise ValueError(f'기존 staging 라벨을 정확히 재현하지 못했습니다: {label}')
        gt=[]
        for line in canonical:
            _,x,y,bw,bh=map(float,line.split())
            gt.append(np.clip([(x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h],[0,0,0,0],[w,h,w,h]).tolist())
        rows.append({'path':str(p),'gt':np.array(gt,dtype=np.float32).reshape(-1,4),'index':idx})
    return rows


def group_bootstrap(c0,c1,groups,n,seed):
    groups=np.asarray(groups); unique=sorted(set(groups)); rng=np.random.default_rng(seed)
    a=np.array([c0[groups==g].sum(0) for g in unique]); b=np.array([c1[groups==g].sum(0) for g in unique])
    delta=[]
    for _ in range(n):
        ix=rng.integers(0,len(unique),len(unique))
        delta.append(operating_metrics(b[ix])['f1']-operating_metrics(a[ix])['f1'])
    return {'delta_f1':operating_metrics(c1)['f1']-operating_metrics(c0)['f1'],
            'ci95':np.quantile(delta,[.025,.975]).tolist(),'resamples':n,
            'unit':'known group','scope':'descriptive OOF interval; CV training sets overlap, not a significance test'}


def recover(run_root,data_root=None,allow_version_drift=False,remeasure_latency=False,reuse_validation_cache=True):
    root=Path(run_root); original_cfg=load_json(root/'cv_config.json')
    source=load_json(root/'source_manifest.json'); folds=load_json(root/'fold_indices.json')['folds']
    if len(folds)!=5: raise ValueError('Expected five folds')
    data_root=data_root or original_cfg['data_root']
    runs=[]
    for fold in folds:
        options=[p for p in (root/f'fold{fold["fold"]}').glob('week5_*') if (p/'test_predictions.json').is_file()]
        if len(options)!=1: raise ValueError(f'fold{fold["fold"]}: 완료 실행 폴더를 하나로 특정할 수 없습니다: {options}')
        run_dir=options[0]
        for name in ['config.json','dataset_manifest.json','cnn_best.pt','detector/weights/best.pt','pip_freeze.txt']:
            if not (run_dir/name).is_file(): raise FileNotFoundError(run_dir/name)
        versions=check_versions(run_dir,allow_version_drift)
        runs.append((fold,run_dir,versions))
    out=root/f'evaluation_repaired_{time.strftime("%Y%m%d_%H%M%S")}_{time.time_ns()%1000000}'
    out.mkdir(exist_ok=False)
    dump(out/'repair_protocol.json',{'source_run':str(root),'training_repeated':False,'max_det':original_cfg['max_det'],
          'fix':'COCO precision tensor at explicit maxDets; invalid overall AP blocks selection',
          'alpha_selection':'original grid and original inner validation only',
          'outer_predictions':'reuse saved boxes, YOLO scores, CNN probabilities',
          'latency':'remeasured' if remeasure_latency else 'not remeasured; original CSV copied separately',
          'allow_version_drift':allow_version_drift,'reuse_validation_cache':reuse_validation_cache,
          'bbox_policy':'GT strictly positive; finite zero-area predictions kept as unmatched detections; nonfinite/reversed coords fail'})
    import shutil
    if (root/'cv_fold_results.csv').is_file(): shutil.copy2(root/'cv_fold_results.csv',out/'original_results_REFERENCE_ONLY.csv')
    all_results=[]; deltas=[]; oof=[None]*len(source); all_counts=[None]*len(source)
    for fold,run_dir,versions in runs:
        k=fold['fold']; dst=out/f'fold{k}'; dst.mkdir()
        cfg=load_json(run_dir/'config.json'); cfg['device']=0 if torch.cuda.is_available() else 'cpu'
        cfg['torch_device']='cuda:0' if torch.cuda.is_available() else 'cpu'
        print(f'Fold {k}/5: 기존 가중치 로드 → inner val 재추론 → alpha 재선택 → 저장된 outer 예측 재평가',flush=True)
        dump(dst/'environment_comparison.json',versions)
        old_trials=[]
        with open(run_dir/'validation_alpha_trials.csv',newline='',encoding='utf-8-sig') as f: old_trials=list(csv.DictReader(f))
        dump(dst/'previous_selection_audit.json',{'trials':old_trials,'invalid_map_trials':sum(float(r['map'])<0 for r in old_trials),
                                                'previous_settings':load_json(run_dir/'locked_validation_settings.json')})
        vr=validation_rows(source,fold['val'],data_root,original_cfg.get('coordinate_eps',1e-6),load_json(run_dir/'dataset_manifest.json'))
        det=YOLO(str(run_dir/'detector/weights/best.pt'))
        checkpoint=torch.load(run_dir/'cnn_best.pt',map_location=cfg['torch_device'],weights_only=True)
        if checkpoint['crop_size']!=cfg['crop_size']: raise ValueError('CNN input size mismatch')
        net=CropCNN().to(cfg['torch_device']); net.load_state_dict(checkpoint['state_dict']); net.eval()
        dump(dst/'weights_hashes.json',{'yolo':sha(run_dir/'detector/weights/best.pt'),'cnn':sha(run_dir/'cnn_best.pt')})
        vb=vq=None
        if reuse_validation_cache:
            expected_hashes=load_json(dst/'weights_hashes.json')
            # Previous successful or interrupted evaluation runs in this original CV run only.
            for cache_path in sorted(root.glob(f'evaluation_repaired_*/fold{k}/validation_predictions.json'),reverse=True):
                if cache_path.parent==dst: continue
                try:
                    if load_json(cache_path.parent/'weights_hashes.json')!=expected_hashes: continue
                    previous_env=load_json(cache_path.parent/'environment_comparison.json')
                    if previous_env['actual']!=versions['actual']: continue
                    cached=load_json(cache_path)
                    cache_by_id={x['source_index']:x for x in cached}
                    if len(cache_by_id)!=len(vr) or set(cache_by_id)!={r['index'] for r in vr}: continue
                    if any(not np.array_equal(np.array(cache_by_id[r['index']]['gt'],dtype=np.float32).reshape(-1,4),r['gt']) for r in vr): continue
                    vb=[{'boxes':np.array(cache_by_id[r['index']]['boxes'],dtype=np.float32).reshape(-1,4),
                         'scores':np.array(cache_by_id[r['index']]['yolo_scores'],dtype=np.float32)} for r in vr]
                    vq=[np.array(cache_by_id[r['index']]['cnn_probs'],dtype=np.float32) for r in vr]
                    if any(len(p['boxes'])!=len(p['scores']) or len(q)!=len(p['scores']) or not np.isfinite(q).all() or np.any((q<0)|(q>1)) for p,q in zip(vb,vq)):
                        vb=vq=None; continue
                    dump(dst/'validation_cache_source.json',{'path':str(cache_path)})
                    print(f'Validation cache 재사용: {cache_path}',flush=True)
                    break
                except (OSError,ValueError,KeyError,TypeError):
                    vb=vq=None; continue
        if vb is None:
            vb,vq=collect(det,net,vr,cfg)
        dump(dst/'validation_predictions.json',[{'source_index':r['index'],'image':r['path'],'gt':r['gt'].tolist(),
            'boxes':p['boxes'].tolist(),'yolo_scores':p['scores'].tolist(),'cnn_probs':q.tolist()} for r,p,q in zip(vr,vb,vq)])
        trials=[{'alpha':a,**map_metrics(rescore(vb,vq,a),vr,cfg)} for a in cfg['alpha_grid']]
        selected=select_alpha(trials); alpha=selected['alpha']; vh=rescore(vb,vq,alpha)
        t0,f0=tune_threshold(vb,vr,cfg); t1,f1=tune_threshold(vh,vr,cfg)
        write_csv(dst/'validation_alpha_trials.csv',trials)
        dump(dst/'locked_validation_settings.json',{'alpha':alpha,'yolo_threshold':t0,'hybrid_threshold':t1,
             'val_f1':[f0,f1],'val_selected_map':selected['map'],'max_det':cfg['max_det']})
        # Only now read outer-holdout predictions; no outer scores enter alpha selection.
        cache=load_json(run_dir/'test_predictions.json')
        indices=[int(Path(p['image']).stem) for p in cache]
        if sorted(indices)!=sorted(fold['test']): raise ValueError('Outer prediction membership mismatch')
        tb=[{'boxes':np.array(p['boxes'],dtype=np.float32).reshape(-1,4),'scores':np.array(p['yolo_scores'],dtype=np.float32)} for p in cache]
        tq=[np.array(p['cnn_probs'],dtype=np.float32) for p in cache]
        if any(len(q)!=len(p['scores']) or not np.isfinite(q).all() or np.any((q<0)|(q>1)) for p,q in zip(tb,tq)):
            raise ValueError('Invalid cached CNN probabilities')
        tr=[{'path':str(Path(data_root)/source[i]['relative_path']),'gt':np.array(p['gt'],dtype=np.float32).reshape(-1,4)} for i,p in zip(indices,cache)]
        th=rescore(tb,tq,alpha); counts=[]; fold_result=[]
        speed=benchmark(det,net,tr,alpha,cfg) if remeasure_latency else None
        for name,p,t in [('YOLOv8',tb,t0),('YOLOv8+CNN',th,t1)]:
            c=counts_at(p,tr,t); counts.append(c)
            result={'fold':k,'model':name,'threshold':t,'alpha':0. if name=='YOLOv8' else alpha,
                    **map_metrics(p,tr,cfg),**operating_metrics(c)}
            if speed: result.update({key:speed[name][key] for key in ['mean_ms','median_ms','p95_ms','fps','samples']})
            fold_result.append(result); all_results.append(result)
        write_csv(dst/'test_comparison.csv',fold_result)
        write_csv(dst/'test_shared_threshold.csv',[{'model':name,'threshold':cfg['shared_threshold'],
                   **operating_metrics(counts_at(p,tr,cfg['shared_threshold']))} for name,p in [('YOLOv8',tb),('YOLOv8+CNN',th)]])
        per=[]; repaired=[]
        for j,(i,r,p,q,hp) in enumerate(zip(indices,tr,tb,tq,th)):
            rec={'source_index':i,'fold':k,'image':r['path'],'gt':r['gt'].tolist(),'boxes':p['boxes'].tolist(),
                 'yolo_scores':p['scores'].tolist(),'cnn_probs':q.tolist(),'hybrid_scores':hp['scores'].tolist()}
            oof[i]=rec; repaired.append(rec); all_counts[i]=[counts[0][j].tolist(),counts[1][j].tolist()]
            per.append({'source_index':i,'image':r['path'],**{f'{n}_{key}':int(counts[m][j,z]) for m,n in enumerate(['yolo','hybrid']) for z,key in enumerate(['tp','fp','fn'])}})
        write_csv(dst/'test_per_image.csv',per); dump(dst/'test_predictions.json',repaired)
        delta={'fold':k,**{f'delta_{m}':fold_result[1][m]-fold_result[0][m] for m in ['map','map_50','precision','recall','f1','fp_per_image']}}
        deltas.append(delta)
        write_csv(out/'cv_fold_results.csv',all_results); write_csv(out/'cv_paired_fold_deltas.csv',deltas)
        dump(out/'completed_folds.json',{'completed':k,'total':5})
        print(f'Fold {k}: alpha={alpha}, mAP={fold_result[0]["map"]:.4f} → {fold_result[1]["map"]:.4f}',flush=True)
        del det,net
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    if any(p is None for p in oof): raise ValueError('Incomplete OOF coverage')
    dump(out/'oof_predictions.json',oof)
    metrics=['map','map_50','precision','recall','f1','fp_per_image']+(['mean_ms','fps'] if remeasure_latency else [])
    summary=[]
    for name in ['YOLOv8','YOLOv8+CNN']:
        for m in metrics:
            vals=np.array([r[m] for r in all_results if r['model']==name])
            summary.append({'model':name,'metric':m,'mean':float(vals.mean()),'std_ddof1':float(vals.std(ddof=1)),'folds':5})
    write_csv(out/'cv_summary_mean_std.csv',summary)
    cs=np.array(all_counts)
    dump(out/'oof_paired_group_bootstrap_f1.json',group_bootstrap(cs[:,0],cs[:,1],[r['group'] for r in source],original_cfg['bootstrap_n'],original_cfg['seed']))
    import matplotlib.pyplot as plt
    fig,axs=plt.subplots(1,2,figsize=(11,4))
    for name in ['YOLOv8','YOLOv8+CNN']:
        r=[r for r in all_results if r['model']==name]
        for ax,m in zip(axs,['map','f1']): ax.plot([x['fold'] for x in r],[x[m] for x in r],marker='o',label=name)
    for ax,title in zip(axs,['Corrected outer mAP50-95 @300','Outer F1 after validation selection']):
        ax.set_xticks(range(1,6)); ax.set_ylim(0,1); ax.set_xlabel('Fold'); ax.set_title(title); ax.legend()
    fig.tight_layout(); fig.savefig(out/'cv_comparison.png',dpi=180); plt.close(fig)
    print('평가 복구 완료:',out); return out,all_results


## 5. 평가기 자체 검증

In [ ]:
import sys, importlib
if '/content' not in sys.path: sys.path.insert(0,'/content')
import week5_coco_fixed, week5_recovery
importlib.reload(week5_coco_fixed)
importlib.reload(week5_recovery)
week5_coco_fixed.self_test()


## 6. 5개 fold 평가 복구

In [ ]:
REPAIRED_OUT, REPAIRED_RESULTS = week5_recovery.recover(
    RUN_ROOT, DATA_ROOT,
    allow_version_drift=ALLOW_VERSION_DRIFT,
    remeasure_latency=REMEASURE_LATENCY,
    reuse_validation_cache=True,
)


## 7. 수정 결과 확인

In [ ]:
import pandas as pd
from IPython.display import display, Image
folds = pd.read_csv(REPAIRED_OUT/'cv_fold_results.csv')
summary = pd.read_csv(REPAIRED_OUT/'cv_summary_mean_std.csv')
display(folds)
display(summary)
display(pd.read_csv(REPAIRED_OUT/'cv_paired_fold_deltas.csv'))
display(Image(filename=str(REPAIRED_OUT/'cv_comparison.png')))
print('수정 결과:',REPAIRED_OUT)


## 필요한 파일과 결과

각 fold의 `detector/weights/best.pt`, `cnn_best.pt`, `test_predictions.json`, `dataset_manifest.json`, `config.json`, `pip_freeze.txt` 및 실행 루트의 `source_manifest.json`, `fold_indices.json`, `cv_config.json`이 필요합니다. 이전 코드가 저장하는 파일들입니다. 원본 이미지와 라벨은 DATA_ROOT에 있어야 합니다. 학습 staging 폴더는 삭제되어 있어도 됩니다.

- 원본 이미지·라벨 hash와 재구성한 val 라벨 hash를 확인합니다. 학습 후 데이터가 바뀌었으면 임의로 계속하지 않습니다.
- 기존 학습 결과는 덮어쓰지 않습니다. `evaluation_repaired_날짜_시간/`에 저장합니다.
- `foldN/previous_selection_audit.json`: 과거 validation AP가 -1이었는지 실제 확인한 기록.
- `foldN/validation_alpha_trials.csv`: 수정된 inner-validation alpha 후보별 AP.
- `foldN/locked_validation_settings.json`: 검증에서 고정한 alpha·threshold.
- `foldN/test_comparison.csv`: 수정된 outer-holdout 비교.
- `cv_summary_mean_std.csv`: 5-fold 평균±표본 표준편차.
- `cv_paired_fold_deltas.csv`, `oof_predictions.json`, `cv_comparison.png`: 차이와 재검토 자료.
- 속도는 기본값에서 재측정하지 않습니다. 기존 속도는 `original_results_REFERENCE_ONLY.csv`에 분리 보존합니다. 기존 경고 출력 비용이 섞였을 수 있고, alpha=0이어도 CNN을 실제 실행한 속도이므로 최종 배포 성능으로 단정하지 마세요. `REMEASURE_LATENCY=True`이면 새 속도를 표에 포함합니다.

alpha=0이 수정 후에도 선택될 수는 있습니다. 이때만 유효한 검증 결과에 근거한 선택으로 해석합니다. 시험 결과를 보고 alpha나 CNN 구조를 다시 선택하지 마세요. 이번 작업은 평가 오류의 복구이며 가중치/분할/후보군은 유지합니다.

원 환경과 패키지 버전이 다르면 원래 환경을 맞추는 것이 우선입니다. ALLOW_VERSION_DRIFT=True는 그 차이를 명시적으로 허용하며 결과 폴더에 차이를 기록합니다. 이 경우 검증 재추론에 버전 차이의 영향이 있을 수 있습니다.

검증: 실제 pycocotools 2.0.11로 기존 maxDets=300/-1 문제를 재현하고, 완전 일치 AP=1, 빈 예측 AP=0, FP 순위 영향, 여러 IoU 평균, 100/300 탐지 상한 차이, 배경 이미지의 FP를 확인했습니다. 제공받은 자료는 결과 텍스트이므로 실제 가중치를 사용한 5-fold 복구는 사용자의 Colab에서 실행해야 합니다.

참고: [COCOeval 공식 구현](https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py), [TorchMetrics 관련 이슈](https://github.com/Lightning-AI/torchmetrics/issues/1153).
